## Models with dict or tuple outputs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/custom_outputs.ipynb)

`WinTSR` needs a callable that returns a single tensor. Plenty of real models don't:
foundation models like `TimeLLM` return a dict of named outputs, and some
architectures return `(predictions, attention_weights)` tuples. **No wrapper class is
needed** — a one-line `lambda` that picks the right field is enough, because `WinTSR`
accepts any callable, not just `nn.Module`.

No dataset download needed.

In [ ]:
%pip install -q tslens
# From source instead:
# %pip install -q "git+https://github.com/khairulislam/tslens.git"

## 1. A model that returns a dict

Stands in for something like `TimeLLM`, which returns named outputs (e.g. a forecast
plus auxiliary text-alignment losses).

In [2]:
import torch
from torch import nn

torch.manual_seed(0)
BATCH, SEQ_LEN, N_FEATURES, PRED_LEN = 8, 30, 4, 5


class DictOutputModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.GRU(N_FEATURES, 32, batch_first=True)
        self.forecast_head = nn.Linear(32, PRED_LEN)
        self.aux_head = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.encoder(x)
        pooled = out.mean(dim=1)
        return {
            "outputs_time": self.forecast_head(pooled),
            "outputs_aux": self.aux_head(pooled),
        }


model = DictOutputModel().eval()
x = torch.randn(BATCH, SEQ_LEN, N_FEATURES)
with torch.no_grad():
    out = model(x)
print({k: tuple(v.shape) for k, v in out.items()})

{'outputs_time': (8, 5), 'outputs_aux': (8, 1)}


Attribution needs a tensor, so wrap the model in a `lambda` that extracts the field
you actually want explained.

In [ ]:
from tslens import WinTSR

inputs = x
baselines = torch.zeros_like(inputs)

attr = WinTSR(lambda x: model(x)["outputs_time"]).attribute(
    inputs=inputs, baselines=baselines, threshold=0.5
)
print("attributions:", tuple(attr.shape), " (batch, pred_len, seq_len, n_features)")

## 2. A model that returns a tuple

Stands in for architectures that also return attention weights or other diagnostics
alongside the prediction.

In [4]:
class TupleOutputModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.GRU(N_FEATURES, 32, batch_first=True)
        self.head = nn.Linear(32, PRED_LEN)

    def forward(self, x):
        out, _ = self.encoder(x)
        predictions = self.head(out.mean(dim=1))
        attention_weights = out.mean(dim=-1)  # stand-in "diagnostic" tensor
        return predictions, attention_weights


model2 = TupleOutputModel().eval()
with torch.no_grad():
    predictions, attention_weights = model2(x)
print("predictions:", tuple(predictions.shape))
print("attention_weights:", tuple(attention_weights.shape))

attr2 = WinTSR(lambda x: model2(x)[0]).attribute(
    inputs=inputs, baselines=baselines, threshold=0.5
)
print("attributions:", tuple(attr2.shape))

predictions: (8, 5)
attention_weights: (8, 30)
attributions: (8, 5, 30, 4)


## 3. Combine with multi-input models

The same trick composes with everything else in this package. A TSlib-style model that
takes four tensors *and* returns a dict just needs both pieces at once:

```python
attr_enc, attr_mark = WinTSR(lambda x_enc, x_mark_enc: model(
    x_enc, x_mark_enc, x_dec, x_mark_dec
)["outputs_time"]).attribute(
    inputs=(x_enc, x_mark_enc),
    baselines=(torch.zeros_like(x_enc), torch.zeros_like(x_mark_enc)),
)
```

Note the lambda's signature changes here: with tuple `inputs`, `WinTSR` calls the
callable positionally with one argument per input tensor, so the wrapper needs a
matching parameter for each.

## Next steps

- **TSlib's four-tensor calling convention.** See the [TSlib models](tslib_models.ipynb)
  notebook for the encoder/decoder split this pattern builds on.
- **Classification outputs.** See the [classification](classification.ipynb) notebook.

Full recipe: [Models that return a dict or a tuple](https://khairulislam.github.io/tslens/integration/#models-that-return-a-dict-or-a-tuple)
in the integration cookbook.